# 🧠 笨笨掃地機｜Agent 原理展示

這本 Notebook 一步一步拆開 Agent 閉環，每一節都對應 `bendan/` 套件裡的一個資料夾：

```
玩家指令 ─► 語氣分析（心情／血壓）＋ 禁區寫入記憶          bendan/gameplay/mood.py、agent/memory.py
         ─► Prompt 組裝（人設＋心情＋世界狀態＋記憶）       bendan/agent/prompts.py、perception.py
         ─► LLM（背景執行緒）                             bendan/llm/
         ─► JSON 解析／白名單驗證（失敗重試一次→發呆）      bendan/agent/parsing.py、agent.py
         ─► 動作佇列執行、突發事件                         bendan/world/game.py、gameplay/incidents.py
         ─► 事件 → 心情／血壓／分數 → 回饋給 LLM            bendan/session.py
```

In [ ]:
# 讓 Notebook 找得到 bendan 套件（不管從專案根目錄或 notebooks/ 執行都可以）
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "bendan").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import HTML, Image, display
import bendan
print("✅ bendan", bendan.__version__, "｜專案位置：", ROOT)
bendan.check_env();   # 課程 JupyterHub 請用「Python 3 (ipykernel)」：它有 ipywidgets（遊戲視窗）、pandas、matplotlib

In [ ]:
from bendan import llm

llm.connect()          # 有 OPENAI_BASE_URL / OPENAI_API_KEY 就用課程 API，沒有就用離線 FakeLLM
llm.list_models();

## 1｜🏠 遊戲世界（只負責「世界」，完全不依賴 LLM）

| 動作 | JSON | 效果 |
|---|---|---|
| 前進 N 格 | `{"op":"forward","n":3}` | 沿目前朝向移動 N 格，途中碰撞即中斷並回報 |
| 左轉 / 右轉 | `turn_left` / `turn_right` | 原地轉 90° |
| 停 | `stop` | 清空動作佇列 |
| 吸 | `suck` | 清掃當前格（若前面有貓……後果自負） |
| 回充 | `return_home` | 自動導航（BFS）回充電座並充電 |

障礙物：🐱 貓、🐶 狗（擋路、會亂跑）、🧱 樂高（踩到會痛到亂轉）、🧦 襪子（吸了會卡住）、🔌 充電線（會纏住）、
💩 狗便便（輾過會抹得滿地都是）、💦 水漬（會滑一格）。

### 1A. 不用 LLM，自己下動作測試引擎

In [ ]:
from bendan.world import Game, DIR_NAMES, run_to_end, LEVELS, check_level
from bendan.render import show, render_ascii

print({k: v["name"] for k, v in LEVELS.items()})
print("關卡檢查：", {k: check_level(k) or "OK" for k in LEVELS})

demo = Game(1)
demo.set_plan([
    {"op": "forward", "n": 1}, {"op": "turn_left"}, {"op": "forward", "n": 2}, {"op": "suck"},
    {"op": "turn_right"}, {"op": "forward", "n": 9},   # 故意走太多格 → 撞牆
])
for e in run_to_end(demo):
    print(f"[{e['type']}] {e['msg']}")
print("位置：", demo.pos, "朝向：", DIR_NAMES[demo.facing], "剩餘灰塵：", len(demo.dirt), f"電量：{demo.battery_pct:.0f}%")
show(demo)

### 1B. 新障礙物：輾過狗便便、踩到水漬

In [ ]:
g4 = Game(4)
print(render_ascii(g4), "\n")
g4.set_plan([{"op": "forward", "n": 5}])      # 前方第 2 格就是便便……
for e in run_to_end(g4):
    print(f"[{e['type']}] {e['msg']}")
print("\n" + render_ascii(g4))
show(g4)

### 1C. 今日挑戰：隨機地圖產生器（同一天種子相同，大家玩到同一張圖）

In [ ]:
from bendan.world.mapgen import daily_level

daily = daily_level()
print(daily["name"], "—", daily["intro"])
show(Game("daily"))

## 2｜🧠 Agent 模組（只負責「大腦」）

| 元件 | 作用 |
|---|---|
| **System Prompt＋人設** | 控制角色個性與輸出格式；「賣蠢模式」vs「正經模式（對照組）」 |
| **心情** | 心情會調整賣蠢強度，並在 Prompt 加上【你的心情】讓 LLM 演出委屈、頂嘴 |
| **世界狀態描述** | 7B 模型座標推理弱，所以改成「以機器人為中心」的相對描述（含電量） |
| **記憶** | 禁區清單、已清掃區域、最近幾句指令與結果 |
| **輸出解析＋白名單驗證** | JSON 解析失敗 / 動作不在選單內 → 自動重試一次 → 仍失敗就「發呆」演出 |

### 2A. System Prompt：動作規格、輸出格式、人設

In [ ]:
from bendan.agent import ACTION_SPEC, PERSONA_SILLY, SILLY_STRENGTH

print(PERSONA_SILLY + SILLY_STRENGTH[2] + "\n\n" + ACTION_SPEC)

### 2B. 世界狀態描述：以機器人為中心

In [ ]:
from bendan.agent import describe_world

print(describe_world(Game(2)))

### 2C. 記憶：玩家說「不准進臥室」會被規則式偵測寫入

In [ ]:
from bendan.agent import Memory, update_rules_from_text

g3 = Game(3)
print(update_rules_from_text(g3, "不准進臥室！"), g3.forbidden)
print(update_rules_from_text(g3, "好啦現在可以進臥室了"), g3.forbidden)
print(Memory().to_text(g3))

### 2D. 輸出解析與白名單驗證

就算模型多講廢話、包了 \`\`\`json，也盡量把 JSON 挖出來；但**動作必須完全在選單內**。`emotion` 是選填，亂寫只會被忽略。

In [ ]:
from bendan.agent import parse_reply

tests = [
    '{"actions": [{"op": "forward", "n": 3}, {"op": "turn_left"}], "say": "好", "emotion": "得意"}',
    '```json\n{"actions": [{"op": "suck"}], "say": "吸！"}\n```',
    '我覺得應該 {"actions": [{"op": "dance"}], "say": "跳舞"}',
    '{"actions": [{"op": "forward", "n": 3},], "say": "多一個逗號"}',
    '{"actions": [], "say": "哼", "emotion": "超級生氣"}',
]
for t in tests:
    print(parse_reply(t))

### 2E. 試一句指令：送給模型的訊息 → 模型原話 → 驗證後的動作 → 執行結果

In [ ]:
from bendan.agent import Agent, actions_to_text
from bendan.gameplay import Mood

g = Game(1)
agent = Agent("silly")
cmd = "往前走一格，然後左轉走兩格，把灰塵吸掉"
mood = Mood(35)    # 假裝笨笨現在有點不爽

print("===== system prompt（人設部分）=====")
print(agent.system_prompt(g, mood).split("【你能用的動作】")[0])
print("===== 送給模型的 user 訊息 =====")
print(agent.build_messages(g, cmd, mood=mood)[1]["content"])

res = agent.think(g, cmd, mood=mood)
print("\n===== 模型原話 =====")
for raw in res["raws"]:
    print(raw)
print("\n格式第一次就正確？", res["format_ok"], "｜有重試？", res["retried"], f"｜延遲 {res['latency']:.2f}s")
print("驗證後動作：", actions_to_text(res["actions"]), "｜情緒：", res["emotion"])
print("🤖 說：", res["say"])
g.set_plan(res["actions"])
for e in run_to_end(g):
    print(f"[{e['type']}] {e['msg']}")
show(g, mood=mood.name, emotion=res["emotion"])

## 3｜🎮 玩法系統

### 3A. 語氣分析 → 心情（靈感：《教妹妹開車》語氣太兇對方會鬧情緒）

刻意用**規則式**分析，而不是再問一次 LLM：結果穩定、可重現、沒有額外延遲，玩家也學得會怎麼哄笨笨。

In [ ]:
import pandas as pd
from bendan.gameplay import analyze_tone

samples = ["請往前走三格然後吸", "笨笨你好棒！", "對不起剛剛太兇了", "笨蛋快點給我前進！！！",
           "白痴 吸啦", "到底在搞什麼", "往前走", "笨笨往前走"]
pd.DataFrame([{"玩家說": s, "心情變化": (t := analyze_tone(s)).delta, "主人血壓": t.bp,
               "標籤": t.describe() or "—", "這次不會被拒絕": t.coaxing} for s in samples])

### 3B. 心情等級 → 賣蠢強度、拒絕機率、Prompt 裡的【你的心情】

In [ ]:
from bendan.gameplay.mood import TIERS

agent = Agent("silly")
g2 = Game(2)   # 第 2 關的基礎賣蠢強度 = 2
rows = []
for v in (90, 60, 35, 10):
    m = Mood(v)
    rows.append({"心情值": v, "等級": f"{m.emoji} {m.name}", "賣蠢強度": agent.effective_silliness(g2, m),
                 "拒絕機率": f"{m.refuse_chance:.0%}", "Prompt": m.prompt_text()})
pd.DataFrame(rows)

### 3C. 主人血壓、突發事件、計分

In [ ]:
import random
from bendan.gameplay import BloodPressure, IncidentDirector, Scoreboard
from bendan.gameplay.pressure import EVENT_BP

print("血壓變化表：", EVENT_BP)
bp = BloodPressure(10)
for t in ["collision", "lego", "sucked_cat", "entered_forbidden"]:
    bp.apply_events([{"type": t}])
    print(f"{t:>18} → {bp.mmhg} mmHg {bp.label}")

# 突發事件：在第 3 關讓笨笨亂走 40 步，看會發生什麼
g = Game(3, seed=7)
director = IncidentDirector(g.cfg["incidents"] + [{"type": "spill", "chance": 0.08, "max": 2}], random.Random(7))
for i in range(40):
    if not g.queue:
        g.set_plan([{"op": random.choice(["turn_left", "turn_right"])}, {"op": "forward", "n": 2}])
    evs = g.step() + director.on_step(g)
    for e in evs:
        if e["type"] in ("spill", "doorbell", "pet_moved"):
            print(f"第 {g.steps:>2} 步 [{e['type']}] {e['msg']}")

s = Scoreboard()
for e in s.on_events([{"type": "cleaned"}, {"type": "cleaned"}, {"type": "cleaned"}, {"type": "lego"}]):
    print(e["msg"])
print("分數明細：", dict(s.breakdown))

### 3D. 恍神與記性差（v3：讓 AI 不再「太聰明」）

試玩回饋：「AI 太聰明，很簡單就過關」。只靠 Prompt 叫 7B 模型偶爾不聽話沒用（它很乖），
所以改成**程式決定**：LLM 給的計畫在執行前，有一定機率被笨笨自己改掉。機率依關卡遞增，**並跟心情連動**——
玩家好好說話，恍神率就會下降，「怎麼說話」因此變成真正的策略。「停」「回充」永遠不會被改。

In [ ]:
import random
from bendan.agent import actions_to_text
from bendan.gameplay.quirks import QUIRKS, apply_quirk, limit_plan, quirk_chance

plan = [{"op": "forward", "n": 2}, {"op": "turn_left"}, {"op": "forward", "n": 1}, {"op": "suck"}]
print("原本的計畫：", actions_to_text(plan))
for kind, name in QUIRKS.items():
    new, msg = apply_quirk(kind, plan, random.Random(0))
    print(f"  {name:<6} → {actions_to_text(new):<22} {msg}")
print("\n記性差（一次只記得 3 個動作）：", actions_to_text(limit_plan(plan, 3)[0]))
cfg = Game(10).cfg
print("第 10 關的恍神率：", {n: f"{quirk_chance(cfg, Mood(v)):.0%}" for n, v in
                         [("開心", 90), ("普通", 60), ("不爽", 35), ("鬧彆扭", 10)]})

### 3E. 方向標示與停電關

黃色箭頭＝正前方、1 2 3＝往前走幾格、左／右／後＝笨笨自己的方向。第 9 關停電：只看得到笨笨附近、走過的地方。

In [ ]:
show(Game(9))
show(Game(9), view="top")

## 4｜🔁 Session：完整閉環＋玩法系統

用程式自動下指令跑完第 0 關（正經模式），看看每回合的事件、最後的結算卡片。
真正遊玩時，遊戲視窗／文字模式就是在做一樣的事，只是多了畫面更新與背景執行緒。

In [ ]:
from bendan.gameplay import Progress
from bendan.session import Session, NullLogger
from bendan.render import hud_html, event_line

demo_progress = Progress(path=ROOT / "saves" / "_demo_progress.json", unlock_all=True)   # 不影響正式存檔
s = Session(0, "serious", logger=NullLogger(), progress=demo_progress, seed=0)
for cmd in ["請往前走三格然後吸", "謝謝！請右轉走兩格再左轉，然後吸"]:
    pre = s.preprocess(cmd)
    events = pre["events"]
    if not pre["handled"]:
        res = s.think(cmd, None, pre["refuse"])
        events += s.apply(cmd, res)
        print(f"🧹 {cmd}\n🤖 {res['say']}")
    while s.busy():
        events += s.tick()
    for e in events:
        print("   ", event_line(e))
display(HTML(hud_html(s, "收工！", [])))
show(s.game)